# Module 04 - Trading Strategy and Simulation

Homework: [https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework4.md](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework4.md)
Submit: [https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw04](https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw04)

The code below is a faithful, function-wrapped port of the official `[2026]_Module_04_Colab_Trading_Simulations.ipynb`
(hyper-parameter tuning skipped, best models hard-coded: DT depth 4/7/10, RF `max_depth=13`, `n_estimators=100`).
Every question runs through `run_scenario()`, so each one starts from a **fresh copy** of the data, as the homework requires.

> **Environment note.** The Random Forest is sensitive to the scikit-learn version. This notebook was executed with
> scikit-learn **1.7.2** (same results as 1.6.x). With 1.8+ / <=1.5 the RF numbers shift (Q1 = 1.119, Q2 = 0.59). Decision-tree results (Q3) do not change.

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

# the course parquet (file id from the official notebook); cached in the gitignored data/raw/
DATA = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()) / "data/raw/stocks_df_combined_2026_09_18.parquet.brotli"
if not DATA.exists():
    import gdown
    gdown.download(id="1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5", output=str(DATA))
df_full = pd.read_parquet(DATA)
print(df_full.shape, df_full.Ticker.nunique(), "tickers")

## Pipeline (port of the official notebook)

* `build_dataset` - sections 0.1-0.2: 2000+ truncation, dummies (`drop_dummies` removes columns from `CATEGORICAL` right before `pd.get_dummies`), 70/15/15 temporal split, optional target redefinition.
* `add_manual_predictions` / `add_model_predictions` - `pred0..pred4` rules, `pred5_clf_10`, `pred6_clf_4`, `pred7..9` (DT depth 7 + thresholds), `pred10/11` (RF thresholds 0.55 / 0.65) and any extra RF thresholds.
* `simulate` - section 3.1.2: $100 per positive prediction, 0.2% fees, 30-day hold, TEST only, `capital = 100 * 30 * Q75(daily trades)`, `CAGR = ((capital + net) / capital) ** (1/4)`.
  As in the official notebook, `net` is summed row by row, so the ~500 most recent rows with unknown 30-day growth (NaN) drop out of it.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

TECHNICAL_INDICATORS = ['adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc',
 'bop', 'cci', 'cmo', 'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext',
 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix',
 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo',
 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk',
 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr',
 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase',
 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine',
 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice']
CUSTOM_NUMERICAL = ['SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative',
                    'volatility', 'ln_volume', 'div_payout', 'stock_split']
MACRO = ['gdppot_us_yoy', 'gdppot_us_qoq', 'cpi_core_yoy', 'cpi_core_mom', 'FEDFUNDS',
         'DGS1', 'DGS5', 'DGS10']
TARGET = 'is_positive_growth_30d_future'


def temporal_split(df, min_date, max_date, train_prop=0.7, val_prop=0.15):
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)
    df['split'] = np.where(df.Date <= train_end, 'train',
                           np.where(df.Date <= val_end, 'validation', 'test'))
    return df


def clean(df):
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    return df


def build_dataset(df_full, drop_dummies=(), target_threshold=None):
    """Steps 0.1-0.2 of the notebook. drop_dummies removes columns from CATEGORICAL
    right before pd.get_dummies (Q1). target_threshold redefines the target (Q3)."""
    df_full = df_full.copy()
    GROWTH = [g for g in df_full.keys() if g.startswith('growth_') and 'future' not in g]
    TECHNICAL_PATTERNS = [g for g in df_full.keys() if 'cdl' in g]
    NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO

    df_full['ln_volume'] = np.log(df_full.Volume + 1e-6)
    df_full['div_payout'] = (df_full.Dividends > 0).astype(int)
    df_full['stock_split'] = (df_full['Stock Splits'] > 0).astype(int)

    if target_threshold is not None:
        df_full[TARGET] = (df_full.growth_future_30d > target_threshold).astype(int)

    df = df_full[df_full.Date >= '2000-01-01'].copy()
    df['Month'] = pd.to_datetime(df['Month'], format='%m').dt.strftime('%B')
    df['Weekday'] = df['Weekday'].astype('string')
    df['wom'] = ((df.Date.dt.day - 1) // 7 + 1).astype(str)
    df['month_wom'] = df.Month + '_w' + df.wom
    del df['wom']

    CATEGORICAL = ['Month', 'Weekday', 'Ticker', 'ticker_type', 'month_wom']
    CATEGORICAL = [c for c in CATEGORICAL if c not in drop_dummies]
    dummies = pd.get_dummies(df[CATEGORICAL], dtype='int32')
    DUMMIES = dummies.keys().to_list()
    new_df = pd.concat([df, dummies], axis=1)
    new_df = temporal_split(new_df, new_df.Date.min(), new_df.Date.max())
    return new_df.copy(), NUMERICAL, DUMMIES


def add_manual_predictions(new_df):
    new_df['pred0_manual_cci'] = (new_df.cci > 200).astype(int)
    new_df['pred1_manual_prev_g1'] = (new_df.growth_30d > 1).astype(int)
    new_df['pred2_manual_prev_g1_and_snp'] = ((new_df.growth_30d > 1) & (new_df.growth_snp500_30d > 1)).astype(int)
    new_df['pred3_manual_dgs10_5'] = ((new_df.DGS10 <= 4.5) & (new_df.DGS5 <= 4)).astype(int)
    new_df['pred4_manual_dgs10_fedfunds'] = ((new_df.DGS10 > 4) & (new_df.FEDFUNDS <= 4.795)).astype(int)


def add_model_predictions(new_df, features, extra_rf_thresholds=()):
    train_valid = new_df[new_df.split.isin(['train', 'validation'])]
    X_tv = clean(train_valid[features].copy())
    y_tv = train_valid[TARGET].fillna(0)
    X_all = clean(new_df[features].copy())

    def dt(depth):
        return DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_tv, y_tv)

    new_df['pred5_clf_10'] = dt(10).predict(X_all)
    new_df['pred6_clf_4'] = dt(4).predict(X_all)
    clf_best = dt(7)
    new_df['pred7_clf_second_best'] = clf_best.predict(X_all)
    p = clf_best.predict_proba(X_all)[:, 1]
    new_df['pred8_clf_second_best_rule_56'] = (p >= 0.56).astype(int)
    new_df['pred9_clf_second_best_rule_66'] = (p >= 0.66).astype(int)

    rf_best = RandomForestClassifier(n_estimators=100, max_depth=13, random_state=42, n_jobs=-1).fit(X_tv, y_tv)
    p_rf = rf_best.predict_proba(X_all)[:, 1]
    new_df['pred10_rf_best_rule_55'] = (p_rf >= 0.55).astype(int)
    new_df['pred11_rf_best_rule_65'] = (p_rf >= 0.65).astype(int)
    for i, t in enumerate(extra_rf_thresholds, start=12):
        new_df[f'pred{i}_rf_best_rule_{round(t * 100)}'] = (p_rf >= t).astype(int)
    return rf_best, p_rf


def simulate(new_df):
    """Section 3.1.2: $100 on every positive prediction, 0.2% fees, TEST only, 30d hold."""
    PREDICTIONS = [k for k in new_df.keys() if k.startswith('pred')]
    test = new_df[new_df.split == 'test']
    rows = []
    for pred in PREDICTIONS:
        sel = test[test[pred] == 1]
        n = len(sel)
        gross_col = 100 * (sel.growth_future_30d - 1)
        gross = gross_col.sum()
        fees = -n * 100 * 0.002
        # as in the notebook: net is summed per row, so rows with unknown future growth (NaN) drop out
        net = (gross_col - 100 * 0.002).sum()
        daily = sel.groupby('Date')[pred].count()
        avg_day = daily.mean()
        q75 = daily.quantile(0.75)
        capital = 100 * 30 * q75
        cagr = ((capital + net) / capital) ** (1 / 4) if n else np.nan
        rows.append((pred, n, gross, fees, net, -fees / gross if gross > 0 else None,
                     net / n if n else None, avg_day, capital, cagr))
    cols = ['prediction', 'sim1_count_investments', 'sim1_gross_rev', 'sim1_fees', 'sim1_net_rev',
            'sim1_fees_percentage', 'sim1_average_net_revenue', 'sim1_avg_investments_per_day',
            'sim1_capital', 'sim1_CAGR']
    res = pd.DataFrame(rows, columns=cols)
    res['sim1_growth_capital_4y'] = (res.sim1_net_rev + res.sim1_capital) / res.sim1_capital
    return res


In [ ]:
def run_scenario(drop_dummies=(), target_threshold=None, extra_rf_thresholds=()):
    """Fresh copy of the whole pipeline -> (new_df, df_sim1_results, RF probabilities for every row)."""
    new_df, NUMERICAL, DUMMIES = build_dataset(df_full, drop_dummies, target_threshold)
    print(f"features: {len(NUMERICAL)} numerical + {len(DUMMIES)} dummies = {len(NUMERICAL) + len(DUMMIES)}")
    add_manual_predictions(new_df)
    _, p_rf = add_model_predictions(new_df, NUMERICAL + DUMMIES, extra_rf_thresholds)
    return new_df, simulate(new_df), p_rf

SHOW = ["prediction", "sim1_count_investments", "sim1_net_rev", "sim1_avg_investments_per_day", "sim1_capital", "sim1_CAGR"]

## Question 1

> **Find the CAGR for the `pred10_rf_best_rule_55` using the new setup (with fewer dummy variables).**
> Remove `Ticker` and `month_wom` from `CATEGORICAL` just before `pd.get_dummies()`. Check: 301 features -> 208.
> Options: 1.121 / 1.126 / 1.136 / 1.141

In [ ]:
# reference: original setup (301 features)
_, res_orig, _ = run_scenario()
print("original setup  pred10 CAGR =", round(res_orig.set_index("prediction").loc["pred10_rf_best_rule_55", "sim1_CAGR"], 4))

In [ ]:
# new setup: no Ticker, no month_wom dummies (208 features)
df_q1, res_q1, _ = run_scenario(drop_dummies=("Ticker", "month_wom"))
res_q1[SHOW]

In [ ]:
q1_cagr = res_q1.set_index("prediction").loc["pred10_rf_best_rule_55", "sim1_CAGR"]
print(f"Q1: CAGR of pred10_rf_best_rule_55 = {q1_cagr:.4f}")
print("best ML predictor by CAGR:", res_q1[res_q1.prediction.str.extract(r"pred(\d+)")[0].astype(int) >= 5].nlargest(2, "sim1_CAGR")[["prediction", "sim1_CAGR"]].values.tolist())

**Answer:** computed **1.1291** with scikit-learn 1.6/1.7 (1.1189 with 1.8+). Submit **1.126** - nearest option, not an exact match (see `answers.md`).

## Question 2

> **What is the probability threshold for the second-best Random Forest decision rule, if you want the number of average trades per day BELOW 5?**
> Add RF predictors for thresholds 0.56 ... 0.64 on top of the Q1 setup (RF `max_depth=13`, `n_estimators=100`).
> Options: 0.54 / 0.56 / 0.57 / 0.58

In [ ]:
thresholds = [round(0.56 + 0.01 * i, 2) for i in range(9)]          # 0.56 ... 0.64
df_q2, res_q2, p_rf = run_scenario(drop_dummies=("Ticker", "month_wom"), extra_rf_thresholds=thresholds)
rf_rules = res_q2[res_q2.prediction.str.contains("rf_best_rule")].copy()
rf_rules["threshold"] = rf_rules.prediction.str.extract(r"rule_(\d+)")[0].astype(int) / 100
rf_rules.sort_values("threshold")[["prediction", "threshold"] + SHOW[1:]]

In [ ]:
# step 1 of the task: tpr/fpr-style table for the Random Forest on TEST (precision up, trades down as the threshold rises)
test = df_q2[df_q2.split == "test"]
y_test, p_test = test["is_positive_growth_30d_future"].fillna(0).astype(int), p_rf[(df_q2.split == "test").to_numpy()]
rows = []
for t in np.round(np.linspace(0.5, 0.66, 9), 2):
    pos = p_test >= t
    tp, fp = int((pos & (y_test == 1)).sum()), int((pos & (y_test == 0)).sum())
    fn = int((~pos & (y_test == 1)).sum())
    rows.append((t, tp + fp, tp / (tp + fp), tp / (tp + fn)))
pd.DataFrame(rows, columns=["threshold", "tp+fp (trades)", "precision", "recall"]).round(4)

In [ ]:
under5 = rf_rules[rf_rules.sim1_avg_investments_per_day < 5].sort_values("sim1_CAGR", ascending=False)
print("rules with avg trades/day < 5, best CAGR first:")
print(under5[["prediction", "threshold", "sim1_avg_investments_per_day", "sim1_CAGR"]].head(3).to_string(index=False))
print("\nQ2 answer (second-best RF rule):", under5.iloc[0].threshold)

**Answer:** **0.57** - the highest-CAGR RF rule with fewer than 5 average trades per day (4.55/day, CAGR 1.103). 0.56 still makes 5.21/day. (With scikit-learn 1.8+ the same table gives 0.59, which is not an offered option.)

## Question 3

> **What is the number of trades for the best predictor `pred5_clf_10` (after redefining the target variable)?**
> `is_positive_growth_30d_future = 1` if `growth_future_30d > 1.022`, else 0. Options: 6 / 7 / 8 / 9

In [ ]:
df_q3, res_q3, _ = run_scenario(target_threshold=1.022)
print("share of class 1 (all rows):", round(df_q3.is_positive_growth_30d_future.mean(), 3))
res_q3[SHOW]

In [ ]:
best = res_q3.loc[res_q3.sim1_CAGR.idxmax()]
r = res_q3.set_index("prediction").loc["pred5_clf_10"]
print("best predictor by CAGR:", best.prediction, round(best.sim1_CAGR, 4))
print(f"pred5_clf_10: {int(r.sim1_count_investments)} trades in 4 years, {r.sim1_avg_investments_per_day:.2f} per day on average, CAGR {r.sim1_CAGR:.4f}")
print("original target, same predictor:", res_orig.set_index("prediction").loc["pred5_clf_10", ["sim1_count_investments", "sim1_avg_investments_per_day", "sim1_CAGR"]].round(3).tolist())

**Answer:** **8** - `pred5_clf_10` makes 7,201 trades on TEST, i.e. **7.85 per day on average** (prints as 8), down from 9.03/day with the original target, at a similar CAGR (1.120 vs 1.119). If the form wants the raw count it is 7,201 (~7 thousand).

## Question 4 - [EXPLORATORY] Describe your ideal trading strategy

Candidate ideas, all backtested on the same TEST window with the same 0.2% fees. **CAGR is the metric** (`((capital + net) / capital) ** (1/4)`, capital = 30 days x Q75 of daily dollars invested, as in `simulate()`).

1. **Conviction-weighted sizing** - $200 on the most confident signals, $100 on the rest.
2. **Ensemble consensus** - trade only when several models agree (here RF + Decision Tree; Logistic Regression / XGBoost are not in the notebook's trained set).
3. **Capital-constrained top-K** - at most K signals per day, ranked by probability.
4. **Stop-loss / take-profit** - exit early at -5% / +10% (daily closes only, no intraday).
5. **CAPM check and leverage** - split profit into market beta and alpha; if the model is *efficient* (positive alpha), double the position, financed at Fed funds + 1%.

In [ ]:
d = df_q2.copy()
d["p_rf"] = p_rf                                   # RF probability for every row (same order as df_q2)
d = d.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# CAPM beta per ticker from daily growth vs S&P 500, estimated on TRAIN+VALIDATION only (no look-ahead)
def beta_of(g):
    g = g.dropna(subset=["growth_1d", "growth_snp500_1d"])
    c = np.cov(g.growth_1d - 1, g.growth_snp500_1d - 1)
    return c[0, 1] / c[1, 1]
beta = d[d.split != "test"].groupby("Ticker").apply(beta_of, include_groups=False)
d["beta"] = d.Ticker.map(beta)
# S&P 500 growth over the same 30 trading days ahead (market leg of CAPM)
d["spx_fut"] = d.Date.map(d.groupby("Date").growth_snp500_30d.first().sort_index().shift(-30))
t = d[d.split == "test"].copy()
print(f"betas: min {beta.min():.2f} | median {beta.median():.2f} | max {beta.max():.2f} | max RF probability on TEST {t.p_rf.max():.3f}")

FEE = 0.002
def trades(sel, usd=100, ret=None):
    r = sel.growth_future_30d - 1 if ret is None else ret
    alpha = r - sel.beta * (sel.spx_fut - 1)       # CAPM alpha of the trade (NaN for the last 30 days: S&P leg unknown)
    return pd.DataFrame({"Date": sel.Date, "usd": usd * np.ones(len(sel)), "pnl": usd * r - usd * FEE,
                         "alpha_pnl": usd * alpha - usd * FEE * (1 + sel.beta), "alpha": alpha}).dropna(subset=["pnl"])

def exposure_capital(tr):                          # 30 days of the busy-day (Q75) dollar exposure
    return 30 * tr.groupby("Date").usd.sum().quantile(0.75)

def cagr(tr, equity=None):
    equity = exposure_capital(tr) if equity is None else equity
    return ((equity + tr.pnl.sum()) / equity) ** 0.25

def stop_exit_returns(mask, stop=-0.05, tp=0.10, trail=False):
    """30-day return of each flagged row with early exits, using next-day closes only."""
    out = pd.Series(np.nan, index=d.index)
    for _, g in d.groupby("Ticker"):
        c, idx = g.Close_x.to_numpy(), g.index.to_numpy()
        for j in np.where(mask[g.index])[0]:
            if j + 30 >= len(c): continue
            path = c[j + 1: j + 31] / c[j] - 1
            ret, peak = path[-1], 0.0
            for r in path:
                peak = max(peak, r)
                if r <= stop or (not trail and r >= tp) or (trail and peak >= tp and r <= peak - 0.05):
                    ret = r; break
            out[idx[j]] = ret
    return out

In [ ]:
flag = lambda m: (d.split == "test") & m
base = t[t.p_rf >= 0.57]
top3_65 = t[t.p_rf >= 0.65].sort_values(["Date", "p_rf"], ascending=[True, False]).groupby("Date").head(3)
r_sl   = stop_exit_returns(flag(d.p_rf >= 0.57).to_numpy(), -0.08, 0.15)
r_lit  = stop_exit_returns(d.index.isin(top3_65.index) & (d.split == "test").to_numpy())

conv = t[t.p_rf >= 0.58]
rows = {
  "0  baseline: RF >= 0.57, $100":                           trades(base),
  "1  conviction: $200 if p>=0.65 else $100 (p>=0.58)":     trades(conv, np.where(conv.p_rf >= 0.65, 200, 100)),
  "2  consensus: RF >= 0.55 AND Decision Tree":             trades(t[(t.p_rf >= 0.55) & (t.pred5_clf_10 == 1)]),
  "3  top-5 per day, p >= 0.60":                            trades(t[t.p_rf >= 0.60].sort_values(["Date", "p_rf"], ascending=[True, False]).groupby("Date").head(5)),
  "4a RF >= 0.57 + stop -8% / take-profit +15%":            trades(d.loc[flag(d.p_rf >= 0.57)], ret=r_sl[flag(d.p_rf >= 0.57)]),
  "4b literal: top-3/day p>=0.65 + stop -5% / TP +10%":     trades(d.loc[top3_65.index], ret=r_lit[top3_65.index]),
}
def alpha_cagr(tr):                                # CAGR of the beta-hedged (market-neutral) P&L, same capital
    a = tr.dropna(subset=["alpha_pnl"])
    return ((exposure_capital(tr) + a.alpha_pnl.sum()) / exposure_capital(tr)) ** 0.25

ideas = pd.DataFrame({k: dict(trades=len(v), per_day=round(len(v) / v.Date.nunique(), 2), CAGR=round(cagr(v), 4),
                              alpha_pct_per_trade=round(100 * v.alpha.mean(), 2), alpha_CAGR=round(alpha_cagr(v), 4))
                      for k, v in rows.items()}).T
ideas

### CAPM: how much of the profit is market beta and how much is alpha?

CAPM says a stock earns `beta x market + alpha`. If the Random Forest only picked high-beta names in a rising market, the signal would just be leveraged market exposure; positive alpha means it genuinely selects better stocks.

In [ ]:
s = base.dropna(subset=["growth_future_30d", "spx_fut", "beta"]).copy()
s["ret"], s["mkt"] = s.growth_future_30d - 1, s.spx_fut - 1
s["alpha"] = s.ret - s.beta * s.mkt
u = t.dropna(subset=["growth_future_30d", "spx_fut", "beta"])
u_alpha = (u.growth_future_30d - 1) - u.beta * (u.spx_fut - 1)
pd.DataFrame({
    "RF >= 0.57 signals": [len(s), s.beta.mean(), 100 * s.ret.mean(), 100 * (s.beta * s.mkt).mean(), 100 * s.alpha.mean()],
    "all stocks (universe)": [len(u), u.beta.mean(), 100 * (u.growth_future_30d - 1).mean(), 100 * (u.beta * (u.spx_fut - 1)).mean(), 100 * u_alpha.mean()],
}, index=["n", "mean beta", "mean 30d return %", "beta x S&P return %", "alpha % (30d)"]).round(2)

In [ ]:
# beta-hedged version: long the stock, short beta x $ of the S&P 500 (hedge also pays fees)
plain = trades(s)
print(f"same signals: unhedged CAGR = {cagr(plain):.4f} | beta-hedged (alpha-only) CAGR = {alpha_cagr(plain):.4f}")

### Leverage: double the position when the model is *efficient*

CAGR is unchanged if capital scales with exposure, so the equity is **fixed at the 1x capital** and the extra $100 is borrowed at `FEDFUNDS + 1%` for the ~42 calendar days of the hold.
"Efficient" is judged only on **already-closed** trades (signals at least 42 days old, last 200): either hit-rate >= 60%, or trailing CAPM alpha > 0.

In [ ]:
sig = base.dropna(subset=["growth_future_30d", "spx_fut", "beta"]).sort_values("Date").copy()
sig["ret"] = sig.growth_future_30d - 1
sig["alpha"] = sig.ret - sig.beta * (sig.spx_fut - 1)
dates, hit, alp = sig.Date.to_numpy(), (sig.ret > 0).to_numpy(), sig.alpha.to_numpy()

def trailing(date, k=200, min_n=50):
    n = np.searchsorted(dates, np.datetime64(date - pd.Timedelta(days=42)), side="right")   # closed signals only
    lo = max(0, n - k)
    return (hit[lo:n].mean(), alp[lo:n].mean()) if n - lo >= min_n else (np.nan, np.nan)

gate = pd.DataFrame([(dt, *trailing(dt)) for dt in sig.Date.unique()], columns=["Date", "trail_hit", "trail_alpha"]).set_index("Date")
sig = sig.join(gate, on="Date")
equity = exposure_capital(trades(sig))           # fixed equity = 1x capital

def levered(L):
    usd = 100 * L
    borrow_cost = (L - 1) * 100 * (sig.FEDFUNDS / 100 + 0.01) * 42 / 365
    return pd.DataFrame({"Date": sig.Date, "usd": usd, "pnl": usd * sig.ret - usd * FEE - borrow_cost})

def mix(L_series):                                # per-trade leverage choice
    lv, one = levered(2), levered(1)
    return one.assign(pnl=np.where(L_series, lv.pnl, one.pnl), usd=np.where(L_series, 200, 100))

variants = {
  "1x (baseline)":                              levered(1),
  "2x always":                                  levered(2),
  "2x when p >= 0.60":                          mix((sig.p_rf >= 0.60).to_numpy()),
  "2x when trailing hit-rate >= 60%":           mix((sig.trail_hit >= 0.60).to_numpy()),
  "2x when trailing CAPM alpha > 0":            mix((sig.trail_alpha > 0).to_numpy()),
}
pd.DataFrame({k: dict(trades=len(v), levered_share=round((v.usd == 200).mean(), 2), CAGR=round(cagr(v, equity), 4)) for k, v in variants.items()}).T

**Reading the tables** (CAGR first, alpha as the second lens; alpha-CAGR = CAGR of the beta-hedged, market-neutral P&L):

* **Baseline RF >= 0.57** - CAGR 1.103, alpha +0.92% per trade (universe: +0.50%). Of its 4.3% average 30-day return, 3.4 points is market beta and 0.9 is alpha: the model does select better stocks, but most of the profit is simply being long. Fully hedged, the CAGR is 1.016.
* **1 Conviction sizing** - CAGR 1.095 vs 1.103 and lower alpha. The idea as written ("> 0.70") never fires: the highest RF probability on TEST is 0.694. Moving the tier to 0.65 does not help.
* **2 Consensus (RF and tree)** - CAGR 1.072, alpha falls to +0.36%: fewer trades, less edge. (Logistic Regression / XGBoost not tested.)
* **3 Top-K per day** - CAGR 1.038: ranking inside a day removes good signals.
* **4 Stop-loss / take-profit** - CAGR 1.073 and **negative alpha (-0.41%)**: stops sell positions that would have recovered. (Approximate: daily closes only, market leg measured over the full 30 days.)
* **5 Leverage** - with equity fixed, 2x on every signal gives CAGR **1.168**. Doubling only when the model is *efficient* gives 1.128 (p >= 0.60), 1.135 (trailing hit-rate >= 60%) and 1.136 (trailing CAPM alpha > 0), i.e. 32-45% of the trades are doubled. Because CAGR ignores risk, "always 2x" wins whenever average return exceeds the borrowing cost; the gate does not raise CAGR, it only switches the leverage off when closed trades stop showing alpha. No margin calls, liquidations or drawdowns are modelled here, and the baseline row of this table (1.1001) uses the 2,047 signals with a known S&P leg, vs 2,118 in the table above (1.1033).

### Creative extensions: Kelly sizing, risk parity, fundamental overlay

All three are compared with their own flat-$100 twin, **with equity fixed at the flat strategy's capital**, so any CAGR gain comes from sizing / filtering and not from simply investing more dollars. CAGR is the main metric; alpha (beta-hedged) is the second lens.

* **Kelly:** among RF >= 0.60 signals, the Kelly fraction `f* = w - (1 - w) / b` uses the win rate `w` and win/loss ratio `b` of the last 200 *already closed* trades (older than 42 days, so no look-ahead). Ticket = $100 x `f*` / expanding median of past `f*` (relative Kelly, capped at 3x); no trade when `f* <= 0`; flat $100 during the 50-trade warm-up.
* **Risk parity:** ticket = $100 x (training-period median volatility) / (stock's own 30-day volatility), clipped to $25-$300, so every position carries similar risk.
* **Fundamental overlay (proxy):** the dataset has **no P/E, free cash flow or sector**, and today's fundamentals would leak the future into a backtest. Proxies from the data itself: *value* = price below its own 250-day average, *quality* = paid a dividend in the last 250 days.

In [ ]:
# extra features for the three strategies (all use only information known on the signal date)
d["vol30"] = d.groupby("Ticker").growth_1d.transform(lambda s: s.rolling(30).std())
d["px_vs_1y"] = d.Close_x / d.groupby("Ticker").Close_x.transform(lambda s: s.rolling(250).mean())
d["div_1y"] = d.groupby("Ticker").div_payout.transform(lambda s: s.rolling(250).sum())
t = d[d.split == "test"].copy()
target_vol = d[d.split != "test"].vol30.median()

def alpha_cagr(tr, equity=None):
    equity = exposure_capital(tr) if equity is None else equity
    return ((equity + tr.dropna(subset=["alpha_pnl"]).alpha_pnl.sum()) / equity) ** 0.25

def summarize(tr, equity=None):
    return dict(trades=len(tr), avg_ticket=round(tr.usd.mean()), CAGR=round(cagr(tr, equity), 4),
                alpha_pct_per_trade=round(100 * tr.alpha.mean(), 2), alpha_CAGR=round(alpha_cagr(tr, equity), 4))

# ---- Kelly
k = t[t.p_rf >= 0.60].dropna(subset=["growth_future_30d"]).sort_values("Date").copy()
k["ret"] = k.growth_future_30d - 1
k_dates, k_rets = k.Date.to_numpy(), k.ret.to_numpy()

def kelly_fraction(date, window=200, min_n=50):
    n = np.searchsorted(k_dates, np.datetime64(date - pd.Timedelta(days=42)), side="right")   # closed trades only
    r = k_rets[max(0, n - window):n]
    wins, losses = r[r > 0], -r[r <= 0]
    if len(r) < min_n or len(wins) == 0 or len(losses) == 0: return np.nan
    w, b = (r > 0).mean(), wins.mean() / losses.mean()
    return w - (1 - w) / b

kf = pd.Series({dt: kelly_fraction(dt) for dt in k.Date.unique()}).sort_index()
ref = kf.expanding().median().shift(1)                       # median of PAST fractions only
rel = (kf / ref).clip(0, 3)
print(f"Kelly f*: median {kf.median():.2f}, share of days with f* <= 0: {(kf <= 0).mean():.1%}, warm-up days: {kf.isna().sum()}")
k["usd"] = k.Date.map(rel).fillna(1.0).to_numpy() * 100        # warm-up / first day -> flat $100
k = k[k.usd > 0]
flat_k = trades(t[t.p_rf >= 0.60])
equity_k = exposure_capital(flat_k)

# ---- Risk parity
def rp_ticket(sel):
    return (100 * target_vol / sel.vol30).clip(25, 300).fillna(100).to_numpy()

top5 = lambda x, thr: x[x.p_rf >= thr].sort_values(["Date", "p_rf"], ascending=[True, False]).groupby("Date").head(5)
rp_a, rp_b, rp_c = top5(t, 0.65), top5(t, 0.57), t[t.p_rf >= 0.57]

# ---- Fundamental overlay (proxy)
f55 = t[t.p_rf >= 0.55]
value, quality = f55.px_vs_1y < 1, f55.div_1y > 0

def row_sized(sel, usd, equity):  # trades() with per-trade tickets
    return summarize(trades(sel, usd), equity)

e_a, e_b, e_c = (exposure_capital(trades(x)) for x in (rp_a, rp_b, rp_c))
e_f = exposure_capital(trades(f55))
creative = {
  "K0 flat $100, RF >= 0.60":                         summarize(flat_k, equity_k),
  "K1 Kelly-sized, RF >= 0.60":                       summarize(trades(k, k.usd.to_numpy()), equity_k),
  "R0 flat, top-5/day p >= 0.65 (literal)":           summarize(trades(rp_a), e_a),
  "R1 risk parity, top-5/day p >= 0.65 (literal)":    row_sized(rp_a, rp_ticket(rp_a), e_a),
  "R2 flat, top-5/day p >= 0.57":                     summarize(trades(rp_b), e_b),
  "R3 risk parity, top-5/day p >= 0.57":              row_sized(rp_b, rp_ticket(rp_b), e_b),
  "R4 flat, all RF >= 0.57":                          summarize(trades(rp_c), e_c),
  "R5 risk parity, all RF >= 0.57":                   row_sized(rp_c, rp_ticket(rp_c), e_c),
  "F0 flat, RF >= 0.55":                              summarize(trades(f55), e_f),
  "F1 + value proxy (price < 1y avg)":                summarize(trades(f55[value]), e_f),
  "F2 + quality proxy (dividend payer)":              summarize(trades(f55[quality]), e_f),
  "F3 + value AND quality":                           summarize(trades(f55[value & quality]), e_f),
}
# ---- Kelly as a regime switch on the plain RF >= 0.57 strategy
b = t[t.p_rf >= 0.57]
b_gate = b[~(b.Date.map(kf) <= 0)]                                   # skip days when trailing Kelly f* <= 0 (warm-up passes)
double = (b_gate.Date.map(rel) >= 1).to_numpy()                      # relative Kelly above its past median -> "model is efficient"
tr_l = trades(b_gate, np.where(double, 200, 100))
borrow = pd.Series(np.where(double, 100 * (b_gate.FEDFUNDS.to_numpy() / 100 + 0.01) * 42 / 365, 0.0), index=b_gate.index)
tr_l["pnl"] -= borrow.loc[tr_l.index]; tr_l["alpha_pnl"] -= borrow.loc[tr_l.index]
creative["K2 flat RF >= 0.57, skip when trailing Kelly <= 0"] = summarize(trades(b_gate), e_c)
creative["K3 K2 + double when relative Kelly >= 1 (borrowed)"] = summarize(tr_l, e_c)
pd.DataFrame(creative).T

### Calendar effect and DCA: buy on Monday, split the $100

Two ideas from the "day-of-the-week effect" literature: (a) only act on a favourable weekday, (b) **dollar-cost average** each signal's $100 into five $20 tranches on five consecutive trading days (each tranche held its own 30 days), which leaves total fees unchanged but averages the entry price. Signals are RF >= 0.57; everything is measured on TEST.

In [ ]:
d["dow"] = d.Date.dt.dayofweek                                  # 0 = Monday
g = d.groupby("Ticker").growth_future_30d
d["g_dca5"] = pd.concat([g.shift(-i) for i in range(5)], axis=1).mean(axis=1, skipna=False)   # five $20 tranches, each held 30 days
t = d[d.split == "test"].copy()
sig = t[t.p_rf >= 0.57]
days = ["Mon", "Tue", "Wed", "Thu", "Fri"]

print("Is there a weekday effect? mean 30d return by entry weekday (%)")
by_day = pd.DataFrame({
    "RF >= 0.57 signals": [100 * (sig[sig.dow == i].growth_future_30d - 1).mean() for i in range(5)],
    "all stocks":         [100 * (t[t.dow == i].growth_future_30d - 1).mean() for i in range(5)],
    "signals (n)":        [(sig.dow == i).sum() for i in range(5)]}, index=days).round(2)
print(by_day.to_string())

fg = d.set_index(["Ticker", "Date"]).growth_future_30d
next_mon = sig.Date + pd.to_timedelta((7 - sig.dow) % 7, unit="D")            # same day if already Monday
g_next_mon = fg.reindex(pd.MultiIndex.from_arrays([sig.Ticker, next_mon])).to_numpy()

mon = sig[sig.dow == 0]
calendar = {
  "D0 every day, lump $100 (baseline)":              summarize(trades(sig), e_c),
  "D1 Monday signals only":                           summarize(trades(mon)),
  "D2 signal any day, buy the next Monday":           summarize(trades(sig, ret=g_next_mon - 1), e_c),
  "D3 DCA: 5 x $20 from the signal day":              summarize(trades(sig, ret=sig.g_dca5 - 1), e_c),
  "D4 DCA: 5 x $20, Monday signals only":             summarize(trades(mon, ret=mon.g_dca5 - 1)),
}
pd.DataFrame(calendar).T

**Reading the creative tests** (CAGR first, alpha second; every row has its flat twin above it):

* **Kelly sizing (K1)** - CAGR 1.056 vs 1.062 flat, but alpha-CAGR 1.026 vs 1.011 and +2.4% vs +1.2% alpha per trade. Most of that comes from the `f* <= 0` switch, which skips ~8% of days: those days' trades are clearly worse (K2 keeps 1,875 of 2,118 trades and lifts alpha per trade from +0.92% to +1.62%). The switch buys alpha *quality*, not CAGR: K2 CAGR 1.092 vs 1.103, K3 (double when relative Kelly >= 1) 1.095. So Kelly works as a **circuit breaker**, not as a growth engine.
* **Risk parity (R1, R3, R5)** - CAGR is slightly *lower* in all three variants (e.g. 1.0995 vs 1.1033) although the average ticket rises to $108-114 (the test years were calmer than the training median). Low-volatility names earned less. Its benefit is lower risk, which CAGR does not measure. The literal version ("top 5 above 0.65") has only 43 trades in 4 years, too thin to conclude anything.
* **Fundamental overlay (F1-F3, proxies)** - the *value* proxy hurts (CAGR 1.083 vs 1.129): the Random Forest leans on momentum and picks stocks that sit *above* their yearly average. The *dividend* quality proxy is almost neutral (1.119). Real P/E and free cash flow are not in the data, so this is a test of the proxies only, not of the idea.
* **Calendar / DCA** - there is no Monday effect here: signals entered on Monday returned 3.7% over 30 days vs 4.3% on average (Friday 5.8%, Wednesday 4.5%, with 400-480 signals per day and overlapping windows, all within noise). Monday-only (1.022) and next-Monday entry (1.098) are worse than any-day entry (1.103). DCA in five tranches gives 1.101, the same as the lump sum: it removes entry-timing luck at no cost to CAGR, which makes it a free execution choice but not an edge.

### Beating the baseline: walk-forward Random Forest on a *relative* target

Everything above wrapped rules around one fixed model and none of it beat the plain threshold, so this section changes the **signal** itself:

* **Walk-forward:** instead of one fit on 2000-2022, retrain every 6 months on all data older than the refit date minus a 45-day gap (the 30-day labels would otherwise leak), from 2016 on.
* **Three targets, same Random Forest** (`max_depth=13`, 100 trees, 208 features): `pos` = "goes up" (the course target), `beat` = "beats the S&P 500 over the same 30 days", `xs` = "ends in the top third of that day's 30-day returns across the 33 stocks".
* **Fair comparison:** each model's threshold is picked on the **validation years (2018-09 to 2022-09)** so that it selects the same share of rows as the baseline (2,118 of 33,163 TEST rows = 6.4%), plus 3% and 10% for robustness. TEST is read afterwards. $100 per signal, 30-day hold, 0.2% fees, as before.

In [ ]:
_, NUMW, DUMW = build_dataset(df_full, ("Ticker", "month_wom"))
XW = d[NUMW + DUMW].replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
gfut = d.growth_future_30d
targets = {
    "pos":  (gfut > 1).astype(float).where(gfut.notna()),
    "beat": (gfut > d.spx_fut).astype(float).where(gfut.notna() & d.spx_fut.notna()),
    "xs":   (gfut.groupby(d.Date).rank(pct=True) >= 0.67).astype(float).where(gfut.notna()),
}
refits = pd.date_range("2016-01-01", d.Date.max(), freq="6MS")

def walk_forward(y):
    p = pd.Series(np.nan, index=d.index)
    for i, R in enumerate(refits):
        R2 = refits[i + 1] if i + 1 < len(refits) else d.Date.max() + pd.Timedelta(days=1)
        tr = (d.Date <= R - pd.Timedelta(days=45)) & y.notna()
        te = (d.Date >= R) & (d.Date < R2)
        if te.sum() == 0: continue
        rf_wf = RandomForestClassifier(n_estimators=100, max_depth=13, random_state=42, n_jobs=-1).fit(XW[tr], y[tr])
        p[te] = rf_wf.predict_proba(XW[te])[:, 1]
    return p

for name, y in targets.items():
    d["p_wf_" + name] = walk_forward(y)
    print(name, "done:", int(d["p_wf_" + name].notna().sum()), "rows scored")

In [ ]:
wv, wt = d[d.split == "validation"], d[d.split == "test"]
share_ref = len(base) / len(t)                     # the baseline picks 6.4% of TEST rows
rows = []
for share in (0.03, share_ref, 0.10, 0.15, 0.20):
    for name in targets:
        col = "p_wf_" + name
        thr = wv[col].quantile(1 - share)          # threshold chosen on VALIDATION only
        v, te_ = trades(wv[wv[col] >= thr]), trades(wt[wt[col] >= thr])
        rows.append(dict(model=name, top_share=round(share, 3), VAL_CAGR=round(cagr(v), 4),
                         TEST_trades=len(te_), per_day=round(len(te_) / te_.Date.nunique(), 2), net_usd=round(te_.pnl.sum()),
                         capital=round(exposure_capital(te_)), TEST_CAGR=round(cagr(te_), 4),
                         alpha_pct=round(100 * te_.alpha.mean(), 2), alpha_CAGR=round(alpha_cagr(te_), 4)))
wf_table = pd.DataFrame(rows)
b_tr = trades(base)
print(f"BASELINE (static RF >= 0.57): {len(b_tr)} trades, {len(b_tr)/b_tr.Date.nunique():.2f}/day, net ${b_tr.pnl.sum():,.0f}, capital ${exposure_capital(b_tr):,.0f}, "
      f"CAGR {cagr(b_tr):.4f}, alpha % {100*b_tr.alpha.mean():.2f}, alpha-CAGR {alpha_cagr(b_tr):.4f}")
wf_table

In [ ]:
# is the gain real? resample entry months with replacement; each strategy keeps ITS OWN capital (as in the CAGR table)
def month_pnl(tr): return tr.groupby(tr.Date.dt.to_period("M")).pnl.sum()
def pick(name, share): return trades(wt[wt["p_wf_" + name] >= wv["p_wf_" + name].quantile(1 - share)])
sx, sp, sx15 = pick("xs", share_ref), pick("pos", share_ref), pick("xs", 0.15)
months = sorted(set(month_pnl(b_tr).index) | set(month_pnl(sx).index) | set(month_pnl(sx15).index))
ix = np.random.default_rng(0).integers(0, len(months), (4000, len(months)))
def boot_cagr(tr):
    m, cap_ = month_pnl(tr).reindex(months, fill_value=0).to_numpy(), exposure_capital(tr)
    return ((cap_ + m[ix].sum(axis=1)) / cap_) ** 0.25
for lab, a, b in [("xs (6.4%) vs static baseline", sx, b_tr), ("xs (15%)  vs static baseline", sx15, b_tr), ("xs (6.4%) vs walk-forward pos", sx, sp)]:
    diff = boot_cagr(a) - boot_cagr(b)
    print(f"{lab:30s} P(first better) = {(diff > 0).mean():.2f} | 95% interval of the CAGR gap [{np.percentile(diff, 2.5):+.3f}, {np.percentile(diff, 97.5):+.3f}]")

yr = lambda z: z.groupby(z.Date.dt.year).agg(trades=("pnl", "size"), net_usd=("pnl", "sum")).round(0).astype(int).T
thr_x = wv.p_wf_xs.quantile(1 - share_ref)
print("\nxs, VALIDATION years:"); print(yr(trades(wv[wv.p_wf_xs >= thr_x])).to_string())
print("xs, TEST years:");       print(yr(sx).to_string())
print(f"xs TEST hit rate {(sx.pnl > 0).mean():.3f}, tickers traded {wt[wt.p_wf_xs >= thr_x].Ticker.nunique()} of {wt.Ticker.nunique()}")

**Reading the walk-forward results** (CAGR first, alpha second; thresholds chosen on validation, TEST read afterwards):

* **The target is what moves the needle.** Same Random Forest and features, retrained every 6 months: "goes up" (`pos`) is weakest (TEST CAGR 1.03-1.08), "beats the S&P" (`beat`) is better (1.07-1.14), and "top third of the day" (`xs`) is best at every selection level (1.15-1.19) and in validation (1.11-1.14 vs ~1.02 for `pos`).
* **Why `xs` CAGR looks so high.** Its signals are sparser (1.8 trades/day at the baseline's 6.4% share), so the reserve is only $6k instead of $18k, and it earns $6.0k vs $8.7k: more return per dollar of capital, not more dollars. The same-capital check is the 20% row: `xs` then trades 4.5/day on the same $18k and earns **$13.6k vs $8.7k** (CAGR 1.151 vs 1.103, alpha-CAGR 1.042 vs 1.016). `beat` at 20% earns $21.7k on $32k (CAGR 1.138).
* **Is it luck?** Resampling the entry months with replacement, `xs` beats the static baseline in 96% of resamples (90% at the 15% share), but the 95% interval of the CAGR gap, [-0.014, +0.183], still touches zero. Against the walk-forward `pos` model the interval excludes zero ([+0.040, +0.237]). So: convincing against "goes up", suggestive but not proven against the static baseline.
* **Consistency.** `xs` makes money in all 5 TEST years (2026 is partial) and in 3 of 5 validation years (2018 and 2022 lose), trades 25 of the 33 tickers, hit rate 58.7%. The walk-forward "goes up" model is itself worse than the static one (1.043 vs 1.103 at the same share), so part of the static baseline's strength is one lucky fit; `pos` is the cleaner like-for-like comparison.

**Answer - strategy: "Relative-strength RF ladder".**

1. **Train a Random Forest to pick winners relative to other stocks, not just "up".** Same model as before (`max_depth=13`, 100 trees, 208 features), but the label is "this stock finishes in the top third of all stocks' 30-day returns that day".
2. **Retrain every 6 months** on past data only (with a 45-day gap so no future information leaks in).
3. **Buy $100** of every stock whose score is above a threshold, picked on the validation years so the number of trades fits the money available.
4. **Hold 30 trading days**, then sell. No stop-loss, one position per ticker.

Why it wins: with the same $18k reserve it earned $13.6k vs $8.7k and CAGR 1.151 vs 1.103 for the baseline (static RF >= 0.57). Picking the day's relative winners beats predicting only "up", because it does not depend on the market going up.